# Full architecture demo

This notebook shows the complete workflow we built:
- standardized data loading
- one `Universe` object with shared prices and returns
- multiple constructions stored on the same universe
- per-universe output folders
- automatic export of market data, constructions, backtests, and Monte Carlo results
- fixed-weight backtesting
- Monte Carlo simulation
- Plotly visualization layer
- optional HTML export of figures
- batch export of all available outputs

## Latest Changes Added

The latest additions that were incorporated are:
- `save_construction(...)` and `save_all_constructions()`
- `save_backtest(...)` and `save_all_backtests()`
- `save_monte_carlo(...)` and `save_all_monte_carlo()`
- `PortfolioVisualizer`
- optional `save_html=True` for plots
- batch plot export methods:
  - `save_all_construction_plots()`
  - `save_all_backtest_plots()`
  - `save_all_monte_carlo_plots()`
  - `save_everything()`
- refined Monte Carlo distribution plot
- refined backtest comparison plot

In [1]:
from pathlib import Path

from portafolios import Universe
from portafolios.data.local_loader import local_loader
from portafolios.constructores import EqualWeightConstructor, Markowitz, NaiveRiskParity
from portafolios.constructores.hrp_style.hrp_core import HRPStyle
from portafolios.eval import Backtester, MonteCarloEngine
from portafolios.plots import PortfolioVisualizer

PROJECT_ROOT = Path.cwd()
CSV_PATH = PROJECT_ROOT / "data_clean_stock_data.csv"

print("Project root:", PROJECT_ROOT)
print("CSV exists:", CSV_PATH.exists())

Project root: c:\Users\narro\Desktop\semestre\honores_actuaria
CSV exists: True


## 1. Create one universe

In [2]:
u = Universe(
    universe_name="thesis_full_demo",
    base_output_dir=PROJECT_ROOT / "outputs" / "runs",
    auto_save_data=True,
    tickers=["AAPL", "MSFT", "AMZN", "GOOG"],
    start="2020-01-01",
    end="2020-12-31",
    loader=local_loader,
    loader_kwargs={"path": CSV_PATH, "prefer_adj_close": True},
    freq="B",
).preparar_datos()

u.prices.head()

,AAPL,MSFT,AMZN,GOOG
Date,,,,
2020-01-02,72.776606,153.428246,94.900497,68.186813
2020-01-03,72.771729,152.683705,94.309998,68.439404
2020-01-06,72.621654,151.872308,95.184502,69.663459
2020-01-07,72.849231,152.416407,95.694504,69.921535
2020-01-08,73.706271,153.495104,95.550003,70.337515


In [3]:
print("Universe name:", u.universe_name)
print("Output dir:", u.output_dir)
print("Data dir:", u.data_dir)
print("Constructions dir:", u.constructions_dir)
print("Backtests dir:", u.backtests_dir)
print("Monte Carlo dir:", u.monte_carlo_dir)
print("Plots dir:", u.plots_dir)
print("Returns shape:", u.returns.shape)
print("Saved prices exists:", (u.data_dir / "prices.csv").exists())
print("Saved returns exists:", (u.data_dir / "returns.csv").exists())
print("Saved metadata exists:", (u.data_dir / "metadata.json").exists())
print("Saved tickers exists:", (u.data_dir / "tickers.json").exists())

Universe name: thesis_full_demo
Output dir: c:\Users\narro\Desktop\semestre\honores_actuaria\results\thesis_full_demo
Data dir: c:\Users\narro\Desktop\semestre\honores_actuaria\results\thesis_full_demo\data
Constructions dir: c:\Users\narro\Desktop\semestre\honores_actuaria\results\thesis_full_demo\constructions
Backtests dir: c:\Users\narro\Desktop\semestre\honores_actuaria\results\thesis_full_demo\backtests
Monte Carlo dir: c:\Users\narro\Desktop\semestre\honores_actuaria\results\thesis_full_demo\monte_carlo
Plots dir: c:\Users\narro\Desktop\semestre\honores_actuaria\results\thesis_full_demo\plots
Returns shape: (260, 4)
Saved prices exists: True
Saved returns exists: True
Saved metadata exists: True
Saved tickers exists: True


## 2. Build multiple constructions on the same universe

In [4]:
construction_kwargs = {
    "ann_factor": 252,
    "construction_start": "2020-01-01",
    "construction_end": "2020-06-30",
}

u.construir(EqualWeightConstructor(), label="equal_weight", **construction_kwargs)
u.construir(Markowitz(), label="markowitz", ret_kind="simple", allow_short=False, **construction_kwargs)
u.construir(NaiveRiskParity(), label="naive_risk_parity", **construction_kwargs)
u.construir(HRPStyle(), label="hrp_style", **construction_kwargs)

u.list_constructions()

['equal_weight', 'markowitz', 'naive_risk_parity', 'hrp_style']

In [5]:
u.compare_insample_metrics()

,method,n_selected,expected_return,volatility,sharpe_m,meta_n_assets,meta_success,meta_message,meta_rf_per_period,meta_objective,...,meta_ret_kind_used,meta_nrp_method,meta_nrp_min_vol,meta_nrp_sigma,meta_hrp_n_clusters,meta_hrp_distance,meta_hrp_clustering,meta_hrp_clusters,meta_hrp_inner_n_assets,meta_hrp_outer_n_assets
name,,,,,,,,,,,,,,,,,,,,,
equal_weight,equal_weight,4,0.472394,0.279167,1.692155,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
hrp_style,HRP-style,4,0.519825,0.287039,1.810992,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,3.0,corr_distance,hierarchical_clusters,"[[MSFT, GOOG], [AAPL], [AMZN]]",1.0,3.0
markowitz,markowitz_max_sharpe,3,0.613606,0.312713,1.962198,4.0,True,Optimization terminated successfully,0.0,0.123607,...,simple,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
naive_risk_parity,Naive Risk Parity (1/sigma),4,0.460697,0.277399,1.660775,NaN,NaN,NaN,NaN,NaN,...,NaN,inverse_vol,1.000000e-08,"{'AAPL': 0.023131691550599295, 'MSFT': 0.01900...",NaN,NaN,NaN,NaN,NaN,NaN


## 3. Evaluate the saved constructions

In [6]:
all_bt = Backtester.run_all(
    u,
    start_date="2020-07-01",
    end_date="2020-12-31",
    ann_factor=252,
    attach=True,
)

all_mc = MonteCarloEngine.run_all(
    u,
    horizon=30,
    n_simulations=500,
    initial_value=1.0,
    attach=True,
    seed=123,
)

{name: u.get_construction(name).backtest_result.summary_metrics for name in u.list_constructions() if u.get_construction(name).backtest_result is not None}

{'equal_weight': {'n_periods': 132,
  'total_return': 0.2501002343812391,
  'annualized_return': 0.5313572050716688,
  'annualized_volatility': 0.25583799600161533,
  'sharpe_ratio': 2.0769284210165324,
  'max_drawdown': -0.16550208519812304},
 'markowitz': {'n_periods': 132,
  'total_return': 0.3247601414658765,
  'annualized_return': 0.7106892217456247,
  'annualized_volatility': 0.3117303603593294,
  'sharpe_ratio': 2.2798203579735326,
  'max_drawdown': -0.1826995330623956},
 'naive_risk_parity': {'n_periods': 132,
  'total_return': 0.24109505462423053,
  'annualized_return': 0.5103665501529422,
  'annualized_volatility': 0.25299344766285464,
  'sharpe_ratio': 2.017311336983989,
  'max_drawdown': -0.16379358358056628},
 'hrp_style': {'n_periods': 132,
  'total_return': 0.2772587492409535,
  'annualized_return': 0.5954972340586258,
  'annualized_volatility': 0.2709406004045916,
  'sharpe_ratio': 2.1978885156723593,
  'max_drawdown': -0.17147300522455056}}

## 4. Export stored objects to disk

In [7]:
u.save_market_data()
u.save_all_constructions()
u.save_all_backtests()
u.save_all_monte_carlo()

{'equal_weight': WindowsPath('c:/Users/narro/Desktop/semestre/honores_actuaria/results/thesis_full_demo/monte_carlo/equal_weight'),
 'markowitz': WindowsPath('c:/Users/narro/Desktop/semestre/honores_actuaria/results/thesis_full_demo/monte_carlo/markowitz'),
 'naive_risk_parity': WindowsPath('c:/Users/narro/Desktop/semestre/honores_actuaria/results/thesis_full_demo/monte_carlo/naive_risk_parity'),
 'hrp_style': WindowsPath('c:/Users/narro/Desktop/semestre/honores_actuaria/results/thesis_full_demo/monte_carlo/hrp_style')}

In [8]:
print((u.get_construction_dir("markowitz") / "weights.csv").exists())
print((u.get_backtest_dir("markowitz") / "portfolio_returns.csv").exists())
print((u.get_mc_dir("markowitz") / "simulated_paths.csv").exists())

True
True
True


## 5. Build the Plotly visualizer

In [9]:
viz = PortfolioVisualizer(u)
viz

## 6. Construction plots

In [10]:
fig_weights_bar = viz.plot_weights_bar("markowitz")
fig_weights_pie = viz.plot_weights_pie("markowitz")
fig_weights_scatter = viz.plot_weights_scatter("markowitz")

fig_weights_bar

## 7. Refined backtest comparison plot

In [11]:
fig_backtest_comparison = viz.plot_backtest_comparison()
fig_backtest_comparison

## 8. Refined Monte Carlo distribution plot

In [12]:
fig_mc_distribution = viz.plot_mc_distribution("markowitz")
fig_mc_distribution

## 9. Save individual figures as HTML

In [13]:
viz.plot_weights_bar("markowitz", save_html=True)
viz.plot_backtest("markowitz", save_html=True)
viz.plot_mc_distribution("markowitz", save_html=True)
viz.plot_backtest_comparison(save_html=True)

In [14]:
print((u.get_construction_dir("markowitz") / "weights_bar.html").exists())
print((u.get_backtest_dir("markowitz") / "backtest.html").exists())
print((u.get_mc_dir("markowitz") / "mc_distribution.html").exists())
print((u.get_plot_dir() / "backtest_comparison.html").exists())

True
True
True
True


## 10. Batch export all Plotly figures

In [15]:
viz.save_all_construction_plots()
viz.save_all_backtest_plots()
viz.save_all_monte_carlo_plots(max_paths=50)

{'equal_weight': ['mc_paths.html', 'mc_distribution.html'],
 'markowitz': ['mc_paths.html', 'mc_distribution.html'],
 'naive_risk_parity': ['mc_paths.html', 'mc_distribution.html'],
 'hrp_style': ['mc_paths.html', 'mc_distribution.html']}

## 11. Batch export absolutely everything

In [16]:
batch_summary = viz.save_everything(max_mc_paths=50)
batch_summary

{'market_data_dir': WindowsPath('c:/Users/narro/Desktop/semestre/honores_actuaria/results/thesis_full_demo/data'),
 'constructions': {'equal_weight': WindowsPath('c:/Users/narro/Desktop/semestre/honores_actuaria/results/thesis_full_demo/constructions/equal_weight'),
  'markowitz': WindowsPath('c:/Users/narro/Desktop/semestre/honores_actuaria/results/thesis_full_demo/constructions/markowitz'),
  'naive_risk_parity': WindowsPath('c:/Users/narro/Desktop/semestre/honores_actuaria/results/thesis_full_demo/constructions/naive_risk_parity'),
  'hrp_style': WindowsPath('c:/Users/narro/Desktop/semestre/honores_actuaria/results/thesis_full_demo/constructions/hrp_style')},
 'backtests': {'equal_weight': WindowsPath('c:/Users/narro/Desktop/semestre/honores_actuaria/results/thesis_full_demo/backtests/equal_weight'),
  'markowitz': WindowsPath('c:/Users/narro/Desktop/semestre/honores_actuaria/results/thesis_full_demo/backtests/markowitz'),
  'naive_risk_parity': WindowsPath('c:/Users/narro/Desktop/s

## 12. Observations and current lacking parts

Main observations on the library right now:
- `Portfolio` vs `Universe` naming is still mixed internally; it works, but a future cleanup could standardize names
- there are still legacy `_init_.py` files in parts of the repo, which can confuse imports
- export system works well, but there is no single manifest/index file yet summarizing all exported artifacts for one universe
- backtesting is intentionally fixed-weight only, which matches your current thesis scope, but there is no transaction cost or benchmark attribution layer yet
- Monte Carlo is currently a simple multivariate normal engine, which is fine for now but limited for heavier-tail modeling
- some older plotting modules still mix plotting with saving/showing, unlike the newer `PortfolioVisualizer` design
- formal automated tests are still missing, so the architecture is working but not yet covered by a test suite